<a href="https://colab.research.google.com/github/muaaz-tahir/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muaaz-tahir/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane (Lane 2) is a mix of ranking and scoring.

It is not classification, because I am not sorting pages into two neat
groups like "good" or "bad". It is not clustering, because I am not
grouping similar pages together.

Instead, I want to give every page a score, then sort all pages from
highest score to lowest score. The team then works down this list, starting
from the top. This is why it is ranking/scoring, not classification or
clustering.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [3]:
!git clone https://github.com/muaaz-tahir/flyrank-ml-internship.git
import pandas as pd

df = pd.read_csv("flyrank-ml-internship/data/raw/content_refresh_anonymized.csv")
print(f"Total rows: {len(df)}")

# Build a proxy target: does this page look like it needs review?
# A page "needs review" if it's declining AND still gets real search demand
df["needs_review"] = ((df["trend_direction"] == "down") & (df["impressions_90d"] >= 100)).astype(int)

print(df["needs_review"].value_counts())
print(f"Share needing review: {df['needs_review'].mean()*100:.1f}%")

fatal: destination path 'flyrank-ml-internship' already exists and is not an empty directory.
Total rows: 30000
needs_review
0    16848
1    13152
Name: count, dtype: int64
Share needing review: 43.8%


I do not have a real column that says "review this page." So I built a
proxy: a page counts as "needs review" if it is trending down AND still
gets at least 100 search impressions in 90 days.

This is not a perfect answer. It is my best guess at what "needs review"
looks like, using signals that already exist in the data. Later I may
improve this proxy as I learn more.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

My metric is precision at the top of the list (precision@50).

Meaning: if I rank all pages and pick my top 50, how many of those 50
actually turn out to be pages that needed review (using my proxy)?

I am not using plain accuracy, because most pages do not need review, so a
model could look "accurate" while still being useless for this job. What
matters is whether my top picks are actually good picks, not the whole list.

As a baseline: if I picked 50 pages completely at random, about 44% of them
would be expected to need review, since that is the overall share in the
data. A useful score should beat 44% clearly.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [4]:
lane_columns = ["content_id", "trend_direction", "impressions_90d",
                 "days_since_last_update", "word_count", "avg_position", "needs_review"]

lane_df = df[lane_columns]
lane_df.head(10)

,content_id,trend_direction,impressions_90d,days_since_last_update,word_count,avg_position,needs_review
0,content_304f48230142,down,3803,20,3221.0,10.6,1
1,content_a1fb4e703a9e,down,15320,25,2481.0,20.3,1
2,content_9aa793d4d895,down,12581,20,3515.0,36.5,1
3,content_331d6c4de07b,stable,11751,22,NaN,6.2,0
4,content_d99b7a2d90ca,down,19140,14,2803.0,44.0,1
5,content_d4084a4bc775,down,3970,20,3080.0,8.5,1
6,content_9a34b442b552,down,20,20,3059.0,7.0,0
7,content_a63219c6e95a,stable,1724,22,NaN,21.2,0
8,content_5e6c160719bc,down,32574,20,3807.0,46.0,1
9,content_c27558df2b0c,down,1240,104,NaN,4.9,1


One row = one web page. Each page has its trend, how many times people
searched and found it, how long since it was last updated, how long the
content is, its average Google position, and my proxy label.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Last week I tried a simple fixed rule: "page is old (180+ days) AND still
popular (500+ impressions)". That rule only found 17 pages out of 30,000.

But my proxy shows 13,152 pages are actually declining with real demand.
The simple rule missed almost all of them, because it only looked at two
signals with hard cutoffs. A model can weigh several signals together
(age, trend, position, word count) instead of using one rigid cutoff,
which is why this is worth more than a fixed rule.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.